<a href="https://colab.research.google.com/github/Sru-j/daily_lab_activities/blob/main/bikes_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#data loading


In [1]:
import os
import glob
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DIR = "/content/drive/My Drive/Colab Notebooks/data/"
os.chdir(DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
os.chdir(DIR)
!ls

 699.csv	      ioc_feed.xml	    superstore.dbf
'alerts (1).json'     netflow.jsonl	    superstore_json.json
 alerts.json	      PlayTennis.csv	    superstore_tsv.tsv
 auth_logs.csv	      PlayTennisTest.csv    superstore.txt
 Bikes.xlsx	      security_policy.yml   superstore_xls.xlsx
 car_dataset.csv      soc_alerts.xlsx	    Titanic_Dataset.csv
 dns_queries.tsv      soc_dump.sql	    unclean_data.csv
 incident_notes.txt   superstore.csv	    web_access.log


In [4]:
datafile='Bikes.xlsx'
filename = DIR+'/'+datafile
df = pd.read_excel(filename)
df

,ID,Marital Status,Gender,Income,Children,Education,Occupation,Home Owner,Cars,Commute Distance,Region,Age,Purchased Bike
0,12496,M,F,40000,1,Bachelors,Skilled Manual,Yes,0,0-1 Miles,Europe,42,No
1,24107,M,M,30000,3,Partial College,Clerical,Yes,1,0-1 Miles,Europe,43,No
2,14177,M,M,80000,5,Partial College,Professional,No,2,2-5 Miles,Europe,60,No
3,24381,S,M,70000,0,Bachelors,Professional,Yes,1,5-10 Miles,Pacific,41,Yes
4,25597,S,M,30000,0,Bachelors,Clerical,No,0,0-1 Miles,Europe,36,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1021,16466,S,F,20000,0,Partial High School,Manual,No,2,0-1 Miles,Europe,32,Yes
1022,19273,M,F,20000,2,Partial College,Manual,Yes,0,0-1 Miles,Europe,63,No
1023,22400,M,M,10000,0,Partial College,Manual,No,1,0-1 Miles,Pacific,26,Yes
1024,20942,S,F,20000,0,High School,Manual,No,1,5-10 Miles,Europe,31,No


#data inspect

In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   marital_status    1000 non-null   object
 1   gender            1000 non-null   object
 2   income            1000 non-null   int64 
 3   children          1000 non-null   int64 
 4   education         1000 non-null   object
 5   occupation        1000 non-null   object
 6   home_owner        1000 non-null   object
 7   cars              1000 non-null   int64 
 8   commute_distance  1000 non-null   int64 
 9   region            1000 non-null   object
 10  age               1000 non-null   int64 
 11  purchased_bike    1000 non-null   object
dtypes: int64(5), object(7)
memory usage: 133.9+ KB


In [5]:
df.columns

Index(['ID', 'Marital Status', 'Gender', 'Income', 'Children', 'Education',
       'Occupation', 'Home Owner', 'Cars', 'Commute Distance', 'Region', 'Age',
       'Purchased Bike'],
      dtype='object')

In [20]:
df.shape

(1026, 13)

In [21]:
df.size

13338

In [8]:
df.dtypes

,0
ID,int64
Marital Status,object
Gender,object
Income,int64
Children,int64
Education,object
Occupation,object
Home Owner,object
Cars,int64
Commute Distance,object


In [9]:
df.isnull().sum()

,0
ID,0
Marital Status,0
Gender,0
Income,0
Children,0
Education,0
Occupation,0
Home Owner,0
Cars,0
Commute Distance,0


In [10]:
df.describe()

,ID,Income,Children,Cars,Age
count,1026.000000,1026.000000,1026.000000,1026.000000,1026.000000
mean,19969.196881,56208.576998,1.892788,1.437622,44.138402
std,5332.672942,31293.284007,1.626670,1.125538,11.349282
min,11000.000000,10000.000000,0.000000,0.000000,25.000000
25%,15304.750000,30000.000000,0.000000,1.000000,35.000000
50%,19744.000000,60000.000000,2.000000,1.000000,43.000000
75%,24457.750000,70000.000000,3.000000,2.000000,52.000000
max,29447.000000,170000.000000,5.000000,4.000000,89.000000


In [13]:
df['Region'].value_counts()

,count
Region,
North America,508
Europe,316
Pacific,202


In [14]:
df['Education'].value_counts()

,count
Education,
Bachelors,311
Partial College,278
High School,184
Graduate Degree,175
Partial High School,78


In [16]:
df['Occupation'].value_counts()

,count
Occupation,
Professional,280
Skilled Manual,259
Clerical,187
Management,174
Manual,126


In [17]:
#check if there is any other values apart from M and F
df['Gender'].value_counts()

,count
Gender,
M,525
F,501


In [18]:
#check if there is any other values apart from M and S
df['Marital Status'].value_counts()

,count
Marital Status,
M,549
S,477


In [19]:
df.duplicated().sum()

np.int64(26)

#data cleaning

In [22]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ','_')
df.columns

Index(['id', 'marital_status', 'gender', 'income', 'children', 'education',
       'occupation', 'home_owner', 'cars', 'commute_distance', 'region', 'age',
       'purchased_bike'],
      dtype='object')

In [23]:
import re

def clean_commute(x):
    x = x.lower().replace('miles', '').strip()

    if '-' in x:  # range like 1-2
        nums = [int(n) for n in x.split('-')]
        return sum(nums) / len(nums)  # average

    elif '+' in x:  # like 10+
        return int(x.replace('+', ''))

    else:  # single value like 1
        return int(re.findall(r'\d+', x)[0])

In [24]:
df['commute_distance'] = df['commute_distance'].apply(clean_commute)

In [25]:
df['commute_distance']

,commute_distance
0,0.5
1,0.5
2,3.5
3,7.5
4,0.5
...,...
1021,0.5
1022,0.5
1023,0.5
1024,7.5


In [26]:
df['commute_distance'].describe()

,commute_distance
count,1026.000000
mean,3.535575
std,3.426005
min,0.500000
25%,0.500000
50%,1.500000
75%,7.500000
max,10.000000


In [27]:
df['commute_distance']=df['commute_distance'].astype(int)
df['commute_distance'].dtypes

dtype('int64')

In [28]:
df.head()

,id,marital_status,gender,income,children,education,occupation,home_owner,cars,commute_distance,region,age,purchased_bike
0,12496,M,F,40000,1,Bachelors,Skilled Manual,Yes,0,0,Europe,42,No
1,24107,M,M,30000,3,Partial College,Clerical,Yes,1,0,Europe,43,No
2,14177,M,M,80000,5,Partial College,Professional,No,2,3,Europe,60,No
3,24381,S,M,70000,0,Bachelors,Professional,Yes,1,7,Pacific,41,Yes
4,25597,S,M,30000,0,Bachelors,Clerical,No,0,0,Europe,36,Yes


#handling dublicates

In [29]:
df.duplicated().sum()

np.int64(26)

In [30]:
# see the duplicates by group, same are grouped together
df[df.duplicated(keep=False)].sort_values(by=list(df.columns))

,id,marital_status,gender,income,children,education,occupation,home_owner,cars,commute_distance,region,age,purchased_bike
12,11434,M,M,170000,5,Partial College,Professional,Yes,0,0,Europe,55,No
1004,11434,M,M,170000,5,Partial College,Professional,Yes,0,0,Europe,55,No
25,12590,S,M,30000,1,Bachelors,Clerical,Yes,0,0,Europe,63,No
1017,12590,S,M,30000,1,Bachelors,Clerical,Yes,0,0,Europe,63,No
17,12610,M,F,30000,1,Bachelors,Clerical,Yes,0,0,Europe,47,No
1009,12610,M,F,30000,1,Bachelors,Clerical,Yes,0,0,Europe,47,No
11,12697,S,F,90000,0,Bachelors,Professional,No,4,10,Pacific,36,No
1003,12697,S,F,90000,0,Bachelors,Professional,No,4,10,Pacific,36,No
5,13507,M,F,10000,2,Partial College,Manual,Yes,0,1,Europe,50,No
1000,13507,M,F,10000,2,Partial College,Manual,Yes,0,1,Europe,50,No


In [31]:
df = df.drop_duplicates()

In [33]:
after = df.shape
after

(1000, 13)

In [36]:
df.duplicated().sum()

np.int64(0)

In [40]:
df.head()

,id,marital_status,gender,income,children,education,occupation,home_owner,cars,commute_distance,region,age,purchased_bike
0,12496,M,F,40000,1,Bachelors,Skilled Manual,Yes,0,0,Europe,42,No
1,24107,M,M,30000,3,Partial College,Clerical,Yes,1,0,Europe,43,No
2,14177,M,M,80000,5,Partial College,Professional,No,2,3,Europe,60,No
3,24381,S,M,70000,0,Bachelors,Professional,Yes,1,7,Pacific,41,Yes
4,25597,S,M,30000,0,Bachelors,Clerical,No,0,0,Europe,36,Yes


In [45]:
# 1st occurnce is at 12, then at 1004, so 1004 is deleted
df.loc[12]
#df.loc[1004]

,12
id,11434
marital_status,M
gender,M
income,170000
children,5
education,Partial College
occupation,Professional
home_owner,Yes
cars,0
commute_distance,0


In [50]:
#deleting id columns
df.drop('id', axis=1, inplace=True)

KeyError: "['id'] not found in axis"

In [51]:
df.loc[12]

,12
marital_status,M
gender,M
income,170000
children,5
education,Partial College
occupation,Professional
home_owner,Yes
cars,0
commute_distance,0
region,Europe


<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   marital_status    1000 non-null   object
 1   gender            1000 non-null   object
 2   income            1000 non-null   int64 
 3   children          1000 non-null   int64 
 4   education         1000 non-null   object
 5   occupation        1000 non-null   object
 6   home_owner        1000 non-null   object
 7   cars              1000 non-null   int64 
 8   commute_distance  1000 non-null   int64 
 9   region            1000 non-null   object
 10  age               1000 non-null   int64 
 11  purchased_bike    1000 non-null   object
dtypes: int64(5), object(7)
memory usage: 133.9+ KB
